# BoneFract — v3 : baselines ResNet-50, DenseNet-121 et Swin-T (même protocole que le notebook v2)

**Objet.** Entraîner trois architectures de référence sur **exactement le même échantillon** (12 000 / 1 500 / 1 500), avec le **même prétraitement, les mêmes augmentations, la même recette d'entraînement et les mêmes graines (42, 123, 2024)** que le notebook v2 (ConvNeXt-Tiny). Les résultats sont ensuite comparés à ConvNeXt-Tiny A (ImageNet, sans attention), déjà entraîné dans `runs_v2/`.

| Clé | Architecture (timm) | Pré-entraînement | Paramètres (≈) | Remarques |
|---|---|---|---|---|
| `resnet50` | `resnet50.tv_in1k` | ImageNet-1k | 25,6 M | BatchNorm ; profondeur stochastique 0,1 |
| `densenet121` | `densenet121.tv_in1k` | ImageNet-1k | 8,0 M | BatchNorm ; **pas de profondeur stochastique** disponible dans timm (dropout de tête 0,2 seul) |
| `swin_t` | `swin_tiny_patch4_window7_224.ms_in22k_ft_in1k` | ImageNet-22k puis 1k | 28,3 M | fenêtre portée de 7 à 12 pour entrer en 384 × 384 ; biais de position relative ré-interpolés par timm |

**Recette (identique à v2 pour tout ce qui n'est pas propre à l'architecture)** : AdamW, learning rate de pointe **2e-5** (×10 pour la couche linéaire finale), échauffement d'une époque puis cosinus, weight decay 0,01 (hors biais, normalisations et biais de position relative), écrêtage du gradient à 1, perte BCE, **EMA des poids 0,999** (le modèle évalué et sélectionné est celui de l'EMA, à déclarer), précision mixte, dropout de tête 0,2, sélection de l'époque sur la **ROC-AUC de validation**, seuil opérationnel fixé sur la validation (rappel cible 90 %).

**Nombre d'époques.** Par défaut **12 époques, patience 4**, comme pour ConvNeXt : c'est ce qui rend la comparaison contrôlée. Le passage à 20 époques / patience 6 se fait en modifiant `CFG["epochs"]` et `CFG["patience"]` ci-dessous, mais il faut alors **ré-entraîner aussi ConvNeXt A** avec la même valeur, sinon les baselines bénéficient d'un budget plus grand que le modèle de référence.

**Limites à déclarer dans l'article** : (i) les pré-entraînements ne sont pas identiques (ConvNeXt et Swin : ImageNet-22k puis 1k ; ResNet et DenseNet : ImageNet-1k seul, aucun poids 22k équivalent dans timm) ; (ii) un seul learning rate (2e-5) pour toutes les architectures, choisi sur ConvNeXt ; il peut être sous-optimal pour les réseaux à BatchNorm ; (iii) les nombres de paramètres diffèrent ; (iv) DenseNet n'a pas de profondeur stochastique.

**Hygiène du test** : `REVEAL_TEST = False` ; le test n'intervient dans aucun choix de développement.

**Mode d'emploi** : cellules dans l'ordre → vérifications et chronométrage → test rapide → (pilote optionnel) → entraînements 3 × 3 → agrégation.


In [ ]:
!pip -q install timm scikit-learn
import os, sys, re, json, time, math, hashlib, zipfile, shutil, random, warnings
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import torch, torch.nn as nn, torch.nn.functional as F, timm
from PIL import Image, ImageFile
from concurrent.futures import ThreadPoolExecutor
from sklearn.metrics import roc_auc_score, average_precision_score, roc_curve
from google.colab import drive
drive.mount('/content/drive')
ImageFile.LOAD_TRUNCATED_IMAGES = True

assert torch.cuda.is_available(), "Active le GPU : Exécution > Modifier le type d'exécution > T4 GPU, puis relance"
dev = torch.device("cuda")
print("torch", torch.__version__, "| timm", timm.__version__, "|", torch.cuda.get_device_name(0))

BASE = Path("/content/drive/MyDrive/Bonefract")
TOP = "BoneFract A Bone Fracture Dataset"
LOCAL = Path("/content/BoneFract")
SEEDS = [42, 123, 2024]

CFG = dict(
    img_size=384, batch_size=32, epochs=12, lr=2e-5, new_lr_mult=10.0, weight_decay=0.01, warmup_epochs=1, min_lr_ratio=0.01,
    grad_clip=1.0, ema_decay=0.999, patience=4, target_recall=0.90, amp=True,
    aug=dict(rotate_deg=10.0, scale=(0.9, 1.1), translate=0.05, brightness=0.2, contrast=0.2),
    mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225),
)
print(json.dumps(CFG, indent=1))
# Le dropout de tête et la profondeur stochastique sont définis par architecture (cellule des modèles).

Mounted at /content/drive


AssertionError: Active le GPU : Exécution > Modifier le type d'exécution > T4 GPU, puis relance

In [ ]:
# --- Utilitaires : métriques, seuil au rappel cible, bootstrap, agrégation, ensemble (testés sur données synthétiques)
import json, math
from pathlib import Path
import numpy as np, pandas as pd
from sklearn.metrics import roc_auc_score, average_precision_score, roc_curve

def metrics_at(y, p, thr):
    """Métriques de classification binaire au seuil thr (prédiction positive si p >= thr)."""
    y = np.asarray(y).astype(int); pred = np.asarray(p) >= thr
    tp = int((pred & (y == 1)).sum()); fp = int((pred & (y == 0)).sum())
    fn = int((~pred & (y == 1)).sum()); tn = int((~pred & (y == 0)).sum())
    rec = tp / max(tp + fn, 1); spe = tn / max(tn + fp, 1); pre = tp / max(tp + fp, 1)
    f1 = 2 * pre * rec / max(pre + rec, 1e-12)
    return {"thr": float(thr), "accuracy": (tp + tn) / len(y), "precision": pre, "recall": rec,
            "specificity": spe, "f1": f1, "tp": tp, "fp": fp, "fn": fn, "tn": tn}

def threshold_for_recall(y, p, target=0.90):
    """Seuil le plus élevé (donc le plus spécifique) atteignant un rappel >= target sur (y, p). À calculer sur la VALIDATION."""
    fpr, tpr, thr = roc_curve(np.asarray(y).astype(int), p, drop_intermediate=False)
    i = int(np.argmax(tpr >= target))
    return float(thr[i])

def bootstrap_ci(y, p, thr, n=1000, seed=0):
    """IC95 % par bootstrap sur les images ; le seuil thr est FIXÉ (non re-sélectionné à chaque tirage)."""
    rng = np.random.RandomState(seed); y = np.asarray(y).astype(int); p = np.asarray(p)
    res = {k: [] for k in ("auc", "recall", "specificity", "precision", "f1")}
    for _ in range(n):
        i = rng.randint(0, len(y), len(y))
        if y[i].min() == y[i].max():
            continue
        res["auc"].append(roc_auc_score(y[i], p[i]))
        m = metrics_at(y[i], p[i], thr)
        for k in ("recall", "specificity", "precision", "f1"):
            res[k].append(m[k])
    return {k: [float(np.percentile(v, 2.5)), float(np.percentile(v, 97.5))] for k, v in res.items()}

def summarize_predictions(y_val, p_val, y_te, p_te, target=0.90, n_boot=1000):
    thr = threshold_for_recall(y_val, p_val, target)
    return {"target_recall": target, "threshold": thr,
            "val": {"auc": float(roc_auc_score(y_val, p_val)), "pr_auc": float(average_precision_score(y_val, p_val)),
                    "at_target": metrics_at(y_val, p_val, thr), "at_0.5": metrics_at(y_val, p_val, 0.5)},
            "test": {"auc": float(roc_auc_score(y_te, p_te)), "pr_auc": float(average_precision_score(y_te, p_te)),
                     "at_target": metrics_at(y_te, p_te, thr), "at_0.5": metrics_at(y_te, p_te, 0.5),
                     "ci_at_target": bootstrap_ci(y_te, p_te, thr, n_boot)}}

def paired_auc_diff(preds_a, preds_b, y, n=1000, seed=0):
    """Différence d'AUC (B - A) moyennée sur les graines, bootstrap APPARIÉ sur les mêmes images de test.
    preds_a / preds_b : listes (une par graine) de vecteurs de probabilités alignés sur y."""
    rng = np.random.RandomState(seed); y = np.asarray(y).astype(int); diffs = []
    for _ in range(n):
        i = rng.randint(0, len(y), len(y))
        if y[i].min() == y[i].max():
            continue
        a = np.mean([roc_auc_score(y[i], p[i]) for p in preds_a])
        b = np.mean([roc_auc_score(y[i], p[i]) for p in preds_b])
        diffs.append(b - a)
    return float(np.mean(diffs)), [float(v) for v in np.percentile(diffs, [2.5, 97.5])]

def aggregate(runs_dir, samp, exps=(1, 2, 3), seeds=(42, 123, 2024), pairs=((1, 2), (2, 3), (1, 3)), n_boot=1000):
    """Tableau moyenne ± écart-type sur les graines, comparaisons appariées et AUC par région (jeu de test)."""
    rows, preds = [], {}
    for e in exps:
        for s in seeds:
            d = Path(runs_dir) / f"exp{e}_seed{s}"
            if not (d / "result.json").exists():
                print(f"manquant : exp{e} graine {s}"); continue
            r = json.load(open(d / "result.json")); preds[(e, s)] = pd.read_csv(d / "test_predictions.csv")
            t = r["test"]
            rows.append({"exp": e, "seed": s, "ROC-AUC": t["auc"], "PR-AUC": t["pr_auc"],
                         **{k: t["at_target"][k] for k in ("accuracy", "precision", "recall", "specificity", "f1")}})
    df = pd.DataFrame(rows)
    if df.empty:
        print("Aucun résultat."); return None, None, None
    metrics = ["ROC-AUC", "PR-AUC", "accuracy", "precision", "recall", "specificity", "f1"]
    g = df.groupby("exp")[metrics]
    summ = g.mean().round(4).astype(str) + " ± " + g.std(ddof=1).round(4).astype(str)
    summ.insert(0, "graines", df.groupby("exp").size())
    print("Jeu de TEST, au seuil déterminé sur la validation (rappel cible) — moyenne ± écart-type sur les graines :")
    print(summ.to_string())

    comps = []
    for a, b in pairs:
        if all((a, s) in preds and (b, s) in preds for s in seeds):
            pa = [preds[(a, s)] for s in seeds]; pb = [preds[(b, s)] for s in seeds]
            assert all((x.rel_path.values == pa[0].rel_path.values).all() for x in pa + pb), "images de test non alignées"
            y = pa[0].label.values
            d_, ci = paired_auc_diff([x.prob.values for x in pa], [x.prob.values for x in pb], y, n=n_boot)
            comps.append({"comparaison": f"exp {b} - exp {a}", "ΔAUC moyen": round(d_, 4), "IC95 %": f"[{ci[0]:.4f} ; {ci[1]:.4f}]",
                          "0 dans l'IC": ci[0] <= 0 <= ci[1]})
    comp = pd.DataFrame(comps)
    if len(comp):
        print("\nComparaisons appariées (bootstrap sur les images de test, AUC moyennée sur les graines) :")
        print(comp.to_string(index=False))

    reg_map = dict(zip(samp.rel_path, samp.region)); reg_rows = []
    for (e, s), tp in preds.items():
        tp = tp.assign(region=tp.rel_path.map(reg_map))
        for reg, gg in tp.groupby("region"):
            if 0 < gg.label.sum() < len(gg):
                reg_rows.append({"exp": e, "région": reg, "n": len(gg), "fractures": int(gg.label.sum()),
                                 "auc": roc_auc_score(gg.label, gg.prob)})
    reg = pd.DataFrame(reg_rows)
    if len(reg):
        reg = reg.groupby(["région", "exp"]).agg(n=("n", "first"), fractures=("fractures", "first"), AUC=("auc", "mean")).round(3).reset_index()
        print("\nAUC par région (test, moyenne sur les graines) ; effectifs de fractures parfois faibles : prudence")
        print(reg.pivot(index="région", columns="exp", values="AUC").to_string())
    return df, comp, reg


def ensemble_report(runs_dir, exps, seeds, target=0.90, n_boot=1000):
    """Moyenne des probabilités des graines, par expérience (ensemble de modèles : à DÉCLARER comme tel).
    Le seuil au rappel cible est fixé sur la validation de l'ensemble ; le test n'intervient dans aucun choix."""
    out = {}
    for e in exps:
        dirs = [Path(runs_dir) / f"exp{e}_seed{s}" for s in seeds]
        if not all((d / "test_predictions.csv").exists() and (d / "valid_predictions.csv").exists() for d in dirs):
            print(f"exp {e} : prédictions incomplètes, ignorée"); continue
        te = [pd.read_csv(d / "test_predictions.csv") for d in dirs]
        va = [pd.read_csv(d / "valid_predictions.csv") for d in dirs]
        assert all((t.rel_path.values == te[0].rel_path.values).all() for t in te), "images de test non alignées"
        assert all((v.rel_path.values == va[0].rel_path.values).all() for v in va), "images de validation non alignées"
        S = summarize_predictions(va[0].label.values, np.mean([v.prob.values for v in va], axis=0),
                                  te[0].label.values, np.mean([t.prob.values for t in te], axis=0), target, n_boot)
        out[e] = S; t = S["test"]; a = t["at_target"]; ci = t["ci_at_target"]
        print(f"Ensemble de {len(seeds)} graines, exp {e} : ROC-AUC test {t['auc']:.4f} [IC95 % {ci['auc'][0]:.4f} - {ci['auc'][1]:.4f}] | "
              f"PR-AUC {t['pr_auc']:.4f} | au seuil {S['threshold']:.4f} (rappel cible {target:.0%} sur la validation) : "
              f"rappel {a['recall']:.3f}, spécificité {a['specificity']:.3f}, précision {a['precision']:.3f}")
    return out

## 1. Données (identiques au notebook v1) : échantillon, extraction, chargement en mémoire

In [ ]:
samp = pd.read_csv(BASE / "echantillon_15000_bonefract.csv")
assert samp.split.value_counts().to_dict() == {"train": 12000, "valid": 1500, "test": 1500}, samp.split.value_counts().to_dict()
# Étiquette du CSV = classe du dossier d'origine
assert (samp.rel_path.str.contains("/Positive/").astype(int) == samp.label).all(), "étiquette du CSV ≠ dossier"
# Aucun fichier identique entre partitions
for a, b in (("train", "valid"), ("train", "test"), ("valid", "test")):
    assert not (set(samp[samp.split == a].md5) & set(samp[samp.split == b].md5)), f"doublon exact {a}/{b}"
print(samp.groupby("split").label.agg(["size", "mean"]).round(3))

LOCAL_ZIP = Path("/content/archive.zip")
SRC_ZIP = BASE / "archive.zip"
# Recopie si le zip local est absent ou incomplet (taille différente de celle du Drive)
if not LOCAL_ZIP.exists() or LOCAL_ZIP.stat().st_size != SRC_ZIP.stat().st_size:
    LOCAL_ZIP.unlink(missing_ok=True)
    shutil.copy(SRC_ZIP, LOCAL_ZIP)

def open_images_zip(path, workdir=Path("/content/_inner")):
    """Ouvre le zip qui contient réellement les PNG (descend dans les zips imbriqués).
    Un zip interne déjà extrait est réutilisé seulement si sa taille est complète."""
    z = zipfile.ZipFile(path)
    names = z.namelist()
    while not any(n.lower().endswith(".png") for n in names):
        info = next(i for i in z.infolist() if i.filename.lower().endswith(".zip"))
        out_file = workdir / info.filename
        if not out_file.exists() or out_file.stat().st_size != info.file_size:
            out_file.unlink(missing_ok=True)          # supprime l'extraction partielle
            print(f"Extraction du zip interne ({info.file_size/1e9:.1f} Go)...")
            z.extract(info, workdir)
        z.close()
        z = zipfile.ZipFile(out_file)
        names = z.namelist()
    return z, set(names)

z, names = open_images_zip(LOCAL_ZIP)

# Préfixe interne éventuel devant "split/région/patient/classe/fichier"
r0 = samp.rel_path.iloc[0]
cand = [n for n in names if n == r0 or n.endswith("/" + r0)]
assert cand, f"{r0} introuvable dans le zip (ex. {sorted(names)[:3]})"
prefix = cand[0][: len(cand[0]) - len(r0)]
print("Préfixe interne du zip :", repr(prefix))

missing = [r for r in samp.rel_path if prefix + r not in names]
assert not missing, f"{len(missing)} images absentes du zip (ex. {missing[:2]})"

todo = [r for r in samp.rel_path if not (LOCAL / TOP / r).exists()]
print("Images à extraire :", len(todo))
for r in todo:
    target = LOCAL / TOP / r
    target.parent.mkdir(parents=True, exist_ok=True)
    with z.open(prefix + r) as src, open(target, "wb") as dst:
        shutil.copyfileobj(src, dst)
z.close()

samp["path"] = [str(LOCAL / TOP / r) for r in samp.rel_path]

def load384(p):
    return np.asarray(Image.open(p).convert("L").resize((CFG["img_size"], CFG["img_size"]), Image.BICUBIC), dtype=np.uint8)

t0 = time.time()
with ThreadPoolExecutor(8) as ex:
    X_all = np.stack(list(ex.map(load384, samp.path, chunksize=64)))
y_all = samp.label.values.astype(np.float32)
idx = {s: np.where(samp.split.values == s)[0] for s in ("train", "valid", "test")}
print(f"Images en mémoire : {X_all.shape} {X_all.dtype} ({X_all.nbytes/1e9:.2f} Go) en {time.time()-t0:.0f} s")
print("Effectifs :", {k: len(v) for k, v in idx.items()})

## 2. Prétraitement, normalisation et augmentations (identiques au notebook v1)

In [ ]:
MEAN = torch.tensor(CFG["mean"], device=dev).view(1, 3, 1, 1)
STD = torch.tensor(CFG["std"], device=dev).view(1, 3, 1, 1)

def augment_batch(x, gen, aug=CFG["aug"]):
    """x : (B,1,H,W) float dans [0,1] sur GPU. Rotation, zoom, translation (une matrice affine par image), luminosité, contraste."""
    B = x.shape[0]
    u = lambda lo, hi: lo + (hi - lo) * torch.rand(B, device=x.device, generator=gen)
    ang = torch.deg2rad(u(-aug["rotate_deg"], aug["rotate_deg"]))
    sc = u(*aug["scale"])                                   # >1 : zoom avant
    tx = u(-aug["translate"], aug["translate"]) * 2         # fraction de la taille -> coordonnées normalisées [-1, 1]
    ty = u(-aug["translate"], aug["translate"]) * 2
    c, s = torch.cos(ang) / sc, torch.sin(ang) / sc
    theta = torch.stack([torch.stack([c, -s, tx], 1), torch.stack([s, c, ty], 1)], 1)    # (B, 2, 3)
    grid = F.affine_grid(theta, list(x.shape), align_corners=False)
    x = F.grid_sample(x, grid, mode="bilinear", padding_mode="zeros", align_corners=False)
    bf = u(1 - aug["brightness"], 1 + aug["brightness"]).view(B, 1, 1, 1)
    cf = u(1 - aug["contrast"], 1 + aug["contrast"]).view(B, 1, 1, 1)
    x = (x * bf).clamp(0, 1)
    m = x.mean(dim=(1, 2, 3), keepdim=True)
    return ((x - m) * cf + m).clamp(0, 1)

def to_input(xb_uint8, gen=None, train=False):
    x = torch.from_numpy(xb_uint8).to(dev, non_blocking=True).float().div_(255).unsqueeze(1)
    if train:
        x = augment_batch(x, gen)
    return (x.repeat(1, 3, 1, 1) - MEAN) / STD

# Contrôles : sortie sans augmentation == simple normalisation ; augmentation reste dans [0, 1]
gen_test = torch.Generator(device=dev).manual_seed(0)
xb = X_all[idx["train"][:8]]
x_raw = torch.from_numpy(xb).to(dev).float().div(255).unsqueeze(1)
x_aug = augment_batch(x_raw, gen_test)
assert x_aug.shape == x_raw.shape and float(x_aug.min()) >= 0 and float(x_aug.max()) <= 1
assert to_input(xb).shape == (8, 3, 384, 384)
print("Contrôles d'augmentation : OK")

In [ ]:
# VALIDATION VISUELLE DES AUGMENTATIONS (exigée par le protocole avant intégration)
gen_vis = torch.Generator(device=dev).manual_seed(123)
sel = idx["train"][np.random.RandomState(0).choice(len(idx["train"]), 6, replace=False)]
x0 = torch.from_numpy(X_all[sel]).to(dev).float().div(255).unsqueeze(1)
versions = [x0] + [augment_batch(x0, gen_vis) for _ in range(3)]
fig, axes = plt.subplots(len(sel), 4, figsize=(11, 2.8 * len(sel)))
for r in range(len(sel)):
    for c in range(4):
        axes[r, c].imshow(versions[c][r, 0].cpu().numpy(), cmap="gray", vmin=0, vmax=1); axes[r, c].axis("off")
        axes[r, c].set_title(("original" if c == 0 else f"augmenté {c}") + f" | {samp.region[sel[r]]} | y={int(y_all[sel[r]])}", fontsize=8)
plt.tight_layout(); plt.show()
print("À vérifier : anatomie préservée, aucun signe de fracture masqué, fond noir aux bords, contraste plausible.")

## 3. Modèles

Trois architectures, une même couche de sortie (1 logit). Seule la couche linéaire finale est réinitialisée (loi normale tronquée σ = 0,02, biais nul, graine fixée) ; elle reçoit un learning rate ×10, comme dans v2. Une cellule de vérification (après la boucle d'entraînement) contrôle la normalisation attendue par les poids, les formes de sortie, le nombre de paramètres « nouveaux », le fonctionnement de l'EMA avec les tampons de BatchNorm, et mesure sur quelques pas la **mémoire GPU** et le **temps par époque** de chaque architecture (utile pour décider du nombre d'époques).


In [ ]:
ARCHS = {
    "resnet50":    dict(name="resnet50.tv_in1k",       kw=dict(drop_rate=0.2, drop_path_rate=0.1), new_params=2049),
    "densenet121": dict(name="densenet121.tv_in1k",    kw=dict(drop_rate=0.2),                     new_params=1025),   # pas de drop_path dans timm
    "swin_t":      dict(name="swin_tiny_patch4_window7_224.ms_in22k_ft_in1k",
                        kw=dict(img_size=CFG["img_size"], window_size=12, drop_rate=0.2, drop_path_rate=0.1), new_params=769),
}
ARCH_KEYS = list(ARCHS)

def reset_head(model, seed):
    """Réinitialise UNIQUEMENT la couche linéaire finale, de façon identique pour une graine donnée."""
    torch.manual_seed(seed)
    fc = model.get_classifier()
    nn.init.trunc_normal_(fc.weight, std=0.02)
    nn.init.zeros_(fc.bias)

def build_model(arch, seed):
    spec = ARCHS[arch]
    m = timm.create_model(spec["name"], pretrained=True, num_classes=1, **spec["kw"])
    reset_head(m, seed)
    ids = {id(p) for p in m.get_classifier().parameters()}
    m._new_names = {n for n, p in m.named_parameters() if id(p) in ids}      # « nouvelles couches » (learning rate majoré)
    return m.to(dev)

count = lambda m: sum(p.numel() for p in m.parameters())
count_new = lambda m: sum(p.numel() for n, p in m.named_parameters() if n in m._new_names)


## 4. Boucle d'entraînement et évaluation

Pour une graine donnée, toutes les architectures voient **les mêmes images dans le même ordre avec les mêmes tirages d'augmentation**. Trois modes : test rapide (`smoke`), pilote du learning rate (`pilot` : validation seule, le test n'est ni calculé ni sauvegardé) et entraînement complet.

In [ ]:
REVEAL_TEST = False   # True uniquement pour l'analyse finale : le test ne doit influencer aucun choix de développement
import copy

def set_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

def make_optimizer(model):
    """AdamW ; poids de dimension >= 2 avec weight decay, biais/normalisations sans ; nouvelles couches avec lr x new_lr_mult."""
    groups = {}
    for n, p in model.named_parameters():
        dec = p.ndim >= 2 and "relative_position_bias_table" not in n      # pas de weight decay sur les biais de position relative (Swin)
        key = ("new" if n in model._new_names else "base", dec)
        groups.setdefault(key, []).append(p)
    pg = [{"params": ps, "weight_decay": CFG["weight_decay"] if dec else 0.0,
           "lr": CFG["lr"] * (CFG["new_lr_mult"] if kind == "new" else 1.0)} for (kind, dec), ps in groups.items()]
    return torch.optim.AdamW(pg, lr=CFG["lr"])

class SimpleEMA:
    """Moyenne mobile exponentielle des poids ; le coefficient monte progressivement au début de l'entraînement."""
    def __init__(self, model, decay):
        self.module = copy.deepcopy(model).eval()
        for p in self.module.parameters():
            p.requires_grad_(False)
        self.decay = decay
        se, sm = self.module.state_dict(), model.state_dict()
        keys = list(se.keys())
        fl = [k for k in keys if se[k].dtype.is_floating_point]
        it = [k for k in keys if not se[k].dtype.is_floating_point]
        self.e = [se[k] for k in fl]; self.m = [sm[k] for k in fl]          # flottants (poids, moyennes/variances BatchNorm) : EMA
        self.e_int = [se[k] for k in it]; self.m_int = [sm[k] for k in it]  # entiers (num_batches_tracked) : simple copie
    @torch.no_grad()
    def update(self, step):
        d = min(self.decay, (1 + step) / (10 + step))
        torch._foreach_mul_(self.e, d)
        torch._foreach_add_(self.e, self.m, alpha=1 - d)
        for a, b in zip(self.e_int, self.m_int):
            a.copy_(b)

@torch.inference_mode()
def predict(model, ids, amp=True, bs=64):
    model.eval(); out = []
    for s in range(0, len(ids), bs):
        with torch.autocast("cuda", dtype=torch.float16, enabled=amp):
            lo = model(to_input(X_all[ids[s:s + bs]]))
        out.append(torch.sigmoid(lo.float()).squeeze(1).cpu().numpy())
    return np.concatenate(out)

def stratified_subset(ids, n):
    h = n // 2
    return np.concatenate([ids[y_all[ids] == 0][:h], ids[y_all[ids] == 1][:h]])

def run_experiment(exp, seed, smoke=False, pilot=False, tag=""):
    root = "runs_base_smoke" if smoke else ("runs_base_pilot" if pilot else "runs_base")
    out_dir = BASE / root / f"exp{exp}_seed{seed}{'_' + tag if tag else ''}"
    label = f"[exp {exp} | graine {seed}{' | ' + tag if tag else ''}]"
    if not smoke and (out_dir / "result.json").exists():
        old = json.load(open(out_dir / "result.json"))
        old_cfg = old.get("cfg", {})
        stale = (old.get("lr") is not None and abs(old["lr"] - CFG["lr"]) > 1e-12) or (old_cfg and (old_cfg.get("epochs") != CFG["epochs"] or old_cfg.get("lr") != CFG["lr"]))
        if stale:
            raise RuntimeError(f"{label} : un résultat existe avec une autre configuration (lr/époques). Supprime {out_dir} ou change le tag.")
        print(f"{label} déjà terminé : ignoré"); return old
    out_dir.mkdir(parents=True, exist_ok=True)
    set_seed(seed)
    tr, va, te = idx["train"], idx["valid"], idx["test"]
    if smoke:
        tr, va, te = stratified_subset(tr, 256), stratified_subset(va, 128), stratified_subset(te, 128)
    epochs = 1 if smoke else CFG["epochs"]; bs = CFG["batch_size"]
    model = build_model(exp, seed); n_params = count(model)
    ema = SimpleEMA(model, CFG["ema_decay"]) if CFG["ema_decay"] else None
    eval_model = ema.module if ema else model
    print(f"{label} lr de pointe {CFG['lr']:g} (nouvelles couches x{CFG['new_lr_mult']:g} : {count_new(model):,} paramètres) | EMA {CFG['ema_decay']} | {epochs} époques max")
    opt = make_optimizer(model)
    spe = math.ceil(len(tr) / bs); total = epochs * spe; warm = CFG["warmup_epochs"] * spe
    def lr_lambda(step):
        if step < warm:
            return (step + 1) / warm
        prog = (step - warm) / max(total - warm, 1)
        return CFG["min_lr_ratio"] + (1 - CFG["min_lr_ratio"]) * 0.5 * (1 + math.cos(math.pi * prog))
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lr_lambda)
    try:
        scaler = torch.amp.GradScaler("cuda", enabled=CFG["amp"])
    except Exception:
        scaler = torch.cuda.amp.GradScaler(enabled=CFG["amp"])
    g_data = torch.Generator().manual_seed(seed)                       # ordre des images : identique entre A et B
    g_aug = torch.Generator(device=dev).manual_seed(seed)              # tirages d'augmentation : identiques entre A et B
    loss_fn = nn.BCEWithLogitsLoss()
    best_auc, best_ep, bad, hist, gstep = -1.0, -1, 0, [], 0
    t_start = time.time()
    for ep in range(epochs):
        model.train(); t0 = time.time(); run_loss = 0.0
        perm = tr[torch.randperm(len(tr), generator=g_data).numpy()]
        for s in range(0, len(perm), bs):
            b = perm[s:s + bs]
            xb = to_input(X_all[b], g_aug, train=True)
            yb = torch.from_numpy(y_all[b]).to(dev)
            with torch.autocast("cuda", dtype=torch.float16, enabled=CFG["amp"]):
                lo = model(xb).squeeze(1)
            loss = loss_fn(lo.float(), yb)
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), CFG["grad_clip"])
            scaler.step(opt); scaler.update(); sched.step()
            if ema:
                ema.update(gstep)
            gstep += 1
            run_loss += loss.item() * len(b)
        assert math.isfinite(run_loss), "perte non finie"
        auc_raw = float(roc_auc_score(y_all[va], predict(model, va))) if ema else None      # modèle brut (journal)
        auc = float(roc_auc_score(y_all[va], predict(eval_model, va)))                       # modèle évalué et sélectionné (EMA si activée)
        hist.append({"epoch": ep + 1, "train_loss": run_loss / len(tr), "val_auc": auc, "val_auc_raw": auc_raw if ema else auc,
                     "lr": opt.param_groups[0]["lr"], "sec": time.time() - t0})
        atxt = ""
        rtxt = f" (brut {auc_raw:.4f})" if ema else ""
        print(f"{label} ép {ep + 1}/{epochs} | perte {run_loss / len(tr):.4f} | AUC val {auc:.4f}{rtxt} | lr {opt.param_groups[0]['lr']:.2e} | {time.time() - t0:.0f} s{atxt}")
        if auc > best_auc + 1e-6:
            best_auc, best_ep, bad = auc, ep + 1, 0
            torch.save(eval_model.state_dict(), out_dir / "best.pt")
        else:
            bad += 1
            if bad >= CFG["patience"]:
                print(f"  arrêt anticipé (patience {CFG['patience']}) ; meilleure époque : {best_ep}"); break
    train_sec = time.time() - t_start
    eval_model.load_state_dict(torch.load(out_dir / "best.pt", map_location=dev))
    pd.DataFrame(hist).to_csv(out_dir / "history.csv", index=False)
    if pilot:      # VALIDATION SEULE : le test n'est ni calculé ni sauvegardé
        p_val = predict(eval_model, va, amp=False)
        res = {"exp": exp, "seed": seed, "tag": tag, "lr": CFG["lr"], "ema": CFG["ema_decay"], "best_epoch": best_ep,
               "best_val_auc": float(roc_auc_score(y_all[va], p_val)), "epochs_run": len(hist), "final_train_loss": hist[-1]["train_loss"],
               "val_auc_curve": [h["val_auc"] for h in hist], "val_auc_raw_curve": [h["val_auc_raw"] for h in hist], "train_sec": train_sec}
        json.dump(res, open(out_dir / "result.json", "w"), indent=1)
        print(f"{label} PILOTE : meilleure époque {best_ep} | AUC val {res['best_val_auc']:.4f} | max brut {max(res['val_auc_raw_curve']):.4f}")
        del model, ema, opt; torch.cuda.empty_cache(); return res
    # Évaluation finale avec le MEILLEUR modèle (sélection sur la validation), en précision normale
    p_val, p_te = predict(eval_model, va, amp=False), predict(eval_model, te, amp=False)
    S = summarize_predictions(y_all[va], p_val, y_all[te], p_te, CFG["target_recall"])
    S.update({"exp": exp, "seed": seed, "smoke": smoke, "n_params": n_params, "best_epoch": best_ep, "best_val_auc_amp": best_auc,
              "epochs_run": len(hist), "train_sec": train_sec, "cfg": CFG, "arch": ARCHS[exp]["name"],
              "versions": {"torch": torch.__version__, "timm": timm.__version__, "gpu": torch.cuda.get_device_name(0)}})
    for name, ids, p in (("valid", va, p_val), ("test", te, p_te)):
        pd.DataFrame({"rel_path": samp.rel_path.values[ids], "label": y_all[ids].astype(int), "prob": p}).to_csv(out_dir / f"{name}_predictions.csv", index=False)
    json.dump(S, open(out_dir / "result.json", "w"), indent=1)
    if smoke:
        (out_dir / "best.pt").unlink(missing_ok=True)
    v = S["val"]["at_target"]
    msg = (f"{label} meilleure époque {best_ep} | VALIDATION : AUC {S['val']['auc']:.4f} | seuil {S['threshold']:.4f} "
           f"-> rappel {v['recall']:.3f}, spécificité {v['specificity']:.3f}")
    if REVEAL_TEST:
        t = S["test"]; a = t["at_target"]
        msg += f" || TEST : AUC {t['auc']:.4f} -> rappel {a['recall']:.3f}, spécificité {a['specificity']:.3f}"
    print(msg)
    del model, ema, opt; torch.cuda.empty_cache()
    return S

def pilot_lr(lrs=(2e-5, 5e-5), exp="resnet50", seed=42):
    """Compare des learning rates de pointe sur la VALIDATION seule (graine 42, une architecture au choix). Journalise EMA et modèle brut."""
    old, res = CFG["lr"], {}
    try:
        for lr in lrs:
            CFG["lr"] = lr
            res[lr] = run_experiment(exp, seed, pilot=True, tag=f"lr{lr:g}")
    finally:
        CFG["lr"] = old
    tab = pd.DataFrame([{"lr": lr, "meilleure époque": r["best_epoch"], "AUC val max (évalué)": round(r["best_val_auc"], 4),
                         "AUC val max (brut)": round(max(r.get("val_auc_raw_curve", r["val_auc_curve"])), 4), "époques": r["epochs_run"],
                         "perte train finale": round(r["final_train_loss"], 3),
                         "écart-type AUC val (instabilité)": round(float(np.std(r["val_auc_curve"][1:])), 4) if len(r["val_auc_curve"]) > 2 else np.nan}
                        for lr, r in res.items()])
    print("\nPilote du learning rate (validation seule) :"); print(tab.to_string(index=False))
    return res

### Vérifications de cohérence et chronométrage

À lancer avant les entraînements : normalisation, formes, nombre de paramètres « nouveaux », EMA avec BatchNorm, mémoire GPU et temps par époque de chaque architecture.


In [ ]:
# VÉRIFICATIONS DE COHÉRENCE (une architecture à la fois pour économiser la mémoire)
from timm.data import resolve_data_config
bench = []
for arch in ARCH_KEYS:
    spec = ARCHS[arch]
    dc = resolve_data_config({}, model=timm.create_model(spec["name"], pretrained=False, **spec["kw"]))
    assert np.allclose(dc["mean"], CFG["mean"]) and np.allclose(dc["std"], CFG["std"]), f"{arch} : normalisation timm ≠ configuration"
    m = build_model(arch, 42)
    assert count_new(m) == spec["new_params"], (arch, count_new(m))
    m.eval()
    with torch.no_grad():
        out = m(torch.randn(2, 3, CFG["img_size"], CFG["img_size"], device=dev))
    assert out.shape == (2, 1), out.shape
    # EMA : doit fonctionner avec les tampons entiers (BatchNorm : num_batches_tracked) et suivre le modèle
    ema = SimpleEMA(m, CFG["ema_decay"])
    w0 = next(p for p in m.parameters() if p.ndim >= 2).detach().clone()
    with torch.no_grad():
        for p in m.parameters():
            p.add_(0.01)
    ema.update(0)
    w_e = next(p for p in ema.module.parameters() if p.ndim >= 2)
    assert not torch.equal(w_e, w0) and not torch.equal(w_e, next(p for p in m.parameters() if p.ndim >= 2)), "EMA inactive"
    del ema

    # Chronométrage : 6 pas d'entraînement (batch réel, précision mixte), 3 premiers ignorés
    m.train(); opt = make_optimizer(m); scaler = torch.amp.GradScaler("cuda"); g = torch.Generator(device=dev).manual_seed(0)
    torch.cuda.reset_peak_memory_stats(); ts = []
    for k in range(6):
        torch.cuda.synchronize(); t0 = time.time()
        xb = to_input(X_all[idx["train"][k * CFG["batch_size"]:(k + 1) * CFG["batch_size"]]], g, train=True)
        yb = torch.from_numpy(y_all[idx["train"][k * CFG["batch_size"]:(k + 1) * CFG["batch_size"]]]).to(dev)
        with torch.autocast("cuda", dtype=torch.float16):
            lo = m(xb).squeeze(1)
        loss = nn.functional.binary_cross_entropy_with_logits(lo.float(), yb)
        opt.zero_grad(set_to_none=True); scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
        torch.cuda.synchronize(); ts.append(time.time() - t0)
    step = float(np.mean(ts[3:])); spe = math.ceil(len(idx["train"]) / CFG["batch_size"])
    bench.append({"architecture": arch, "paramètres (M)": round(count(m) / 1e6, 1), "pic mémoire GPU (Go)": round(torch.cuda.max_memory_allocated() / 1e9, 1),
                  "s/pas": round(step, 3), "≈ min/époque (entraînement seul)": round(step * spe / 60, 1)})
    del m, opt, scaler; torch.cuda.empty_cache()
tab = pd.DataFrame(bench)
tab["≈ h pour 3 graines, %d époques (+10 %% évaluation)" % CFG["epochs"]] = (tab["≈ min/époque (entraînement seul)"] * CFG["epochs"] * 3 * 1.1 / 60).round(1)
print(tab.to_string(index=False))
print("Vérifications OK. Si un modèle sature la mémoire du GPU, ne change PAS la taille de lot sans le déclarer : elle fait partie du protocole.")


## 5. Test rapide (obligatoire avant tout)

256 images d'entraînement, 128 de validation, 128 de test, 1 époque, graine 42, pour chaque architecture. Il ne sert qu'à vérifier que tout tourne de bout en bout ; ses métriques n'ont aucune signification. Écrit dans `runs_base_smoke/`.


In [ ]:
smoke = [run_experiment(a, 42, smoke=True) for a in ARCH_KEYS]
tab = pd.DataFrame([{"architecture": r["exp"], "paramètres": r["n_params"], "AUC val (128 images, sans signification)": round(r["val"]["auc"], 3), "secondes": round(r["train_sec"])} for r in smoke])
print(tab.to_string(index=False))
print("\nContrôle : l'exécution doit aller au bout pour les trois architectures (chargement, entraînement, EMA, évaluation, sauvegardes).")


## 6. Pilote du learning rate (OPTIONNEL, validation seule)

Par défaut, le protocole garde **le même learning rate (2e-5) pour toutes les architectures**, ce qui est le plus simple à défendre dans l'article (« recette identique »). Les réseaux à BatchNorm peuvent toutefois préférer un learning rate plus élevé. Si tu as le temps (≈ 40 min par essai et par architecture), décommente la cellule ci-dessous : elle compare 2e-5 et 5e-5 sur la graine 42, **sur la validation seule**. Si un learning rate différent est retenu pour une architecture, il faut le **déclarer** et ne pas le choisir en regardant le test.


In [ ]:
# Optionnel (ne pas lancer par défaut) :
# pilot_rn = pilot_lr((2e-5, 5e-5), exp="resnet50")
# pilot_dn = pilot_lr((2e-5, 5e-5), exp="densenet121")


## 7. Entraînements complets : 3 architectures × 3 graines

Chaque entraînement terminé écrit `runs_base/exp{architecture}_seed{graine}/result.json` ; relancer après une déconnexion saute ceux déjà faits. Ordre : graine par graine, pour obtenir tôt une comparaison complète des trois architectures. Un `RuntimeError` signale un résultat existant obtenu avec un autre learning rate ou un autre nombre d'époques (pour ne jamais mélanger deux protocoles).


In [ ]:
print("Learning rate de pointe :", CFG["lr"], "| nouvelles couches :", CFG["lr"] * CFG["new_lr_mult"], "| époques :", CFG["epochs"], "| patience :", CFG["patience"])
RUNS = [(a, s) for s in SEEDS for a in ARCH_KEYS]
results = {}
for a, s in RUNS:
    results[(a, s)] = run_experiment(a, s)


## 8. Agrégation et comparaison avec ConvNeXt-Tiny A

Les résultats de ConvNeXt-Tiny A (`runs_v2/expA_seed*`) sont copiés (fichiers de résultats et de prédictions uniquement) dans `runs_base/expconvnext_seed*` pour être agrégés avec les trois baselines. **La cellule vérifie d'abord que ConvNeXt A a été entraîné avec le même learning rate et le même nombre d'époques** ; sinon la comparaison n'est pas contrôlée. Les comparaisons appariées (bootstrap sur les images de test) mesurent l'incertitude liée au jeu de test, pas celle de l'entraînement : un écart inférieur à environ 0,01 d'AUC reste inattribuable avec 3 graines.


In [ ]:
import shutil
RB, RV = BASE / "runs_base", BASE / "runs_v2"
for s in SEEDS:
    src, dst = RV / f"expA_seed{s}", RB / f"expconvnext_seed{s}"
    if not (src / "result.json").exists():
        print(f"ConvNeXt A graine {s} : introuvable dans runs_v2"); continue
    cfg_c = json.load(open(src / "result.json")).get("cfg", {})
    for k in ("lr", "epochs", "patience", "ema_decay", "batch_size", "weight_decay"):
        if cfg_c.get(k) != CFG[k]:
            print(f"⚠ ConvNeXt A graine {s} : {k} = {cfg_c.get(k)} ≠ baselines ({CFG[k]}) -> comparaison NON contrôlée")
    dst.mkdir(parents=True, exist_ok=True)
    for f in ("result.json", "test_predictions.csv", "valid_predictions.csv"):
        shutil.copy(src / f, dst / f)
ALL = ("convnext",) + tuple(ARCH_KEYS)
df_runs, df_comp, df_reg = aggregate(RB, samp, exps=ALL, pairs=tuple(("convnext", a) for a in ARCH_KEYS))
ens = ensemble_report(RB, ALL, SEEDS)


In [ ]:
import numpy as np, pandas as pd
from pathlib import Path
from sklearn.metrics import roc_auc_score

def region_report(dirs, samp, split="valid", n_boot=500, seed=0, keep_regions=None):
    """AUC par région, AUC globale et AUC moyenne des régions (macro), pour UN modèle ou un ENSEMBLE (moyenne des probabilités des dossiers).
    split='valid' par défaut : les décisions de périmètre se prennent sur la validation. Mettre 'test' seulement pour l'évaluation finale.
    keep_regions : liste de régions pour restreindre le périmètre (analyse secondaire déclarée)."""
    preds = [pd.read_csv(Path(d) / f"{split}_predictions.csv") for d in dirs]
    assert all((p.rel_path.values == preds[0].rel_path.values).all() for p in preds), "images non alignées entre les dossiers"
    df = preds[0][["rel_path", "label"]].copy()
    df["prob"] = np.mean([p.prob.values for p in preds], axis=0)
    df["region"] = df.rel_path.map(dict(zip(samp.rel_path, samp.region)))
    assert df.region.notna().all(), "région introuvable pour certaines images"
    if keep_regions is not None:
        df = df[df.region.isin(keep_regions)]
    rng = np.random.RandomState(seed); rows = []
    for reg, g in df.groupby("region"):
        y, p = g.label.values, g.prob.values
        boots = []
        for _ in range(n_boot):
            i = rng.randint(0, len(g), len(g))
            if 0 < y[i].sum() < len(i):
                boots.append(roc_auc_score(y[i], p[i]))
        rows.append({"région": reg, "n": len(g), "fractures": int(y.sum()), "prévalence": round(float(y.mean()), 3),
                     "AUC": roc_auc_score(y, p), "IC95 % bas": np.percentile(boots, 2.5), "IC95 % haut": np.percentile(boots, 97.5)})
    tab = pd.DataFrame(rows)
    pooled = roc_auc_score(df.label, df.prob)
    macro = tab["AUC"].mean()
    print(f"Jeu : {split} | modèles moyennés : {len(preds)} | images : {len(df)}")
    print(tab.round(3).to_string(index=False))
    print(f"\nAUC globale (toutes images mélangées) : {pooled:.4f}")
    print(f"AUC moyenne des régions (macro)        : {macro:.4f}")
    print("-> si l'AUC globale dépasse nettement la moyenne par région, une partie du score global vient des DIFFÉRENCES ENTRE RÉGIONS (prévalence, échelle des scores), pas de la détection au sein d'une région.")
    return tab, pooled, macro

# Exemple : Swin-T, ensemble des 3 graines, sur la VALIDATION (adapte les dossiers à ce qui existe sur ton Drive)
dirs = [BASE / "runs_base" / f"expswin_t_seed{s}" for s in SEEDS]
tab_all, pooled_all, macro_all = region_report(dirs, samp)
# Périmètre restreint (analyse secondaire, critère à fixer d'après les AUC de validation ci-dessus) :
# tab_sub, pooled_sub, macro_sub = region_report(dirs, samp, keep_regions=["arm", "lower leg", "thigh", "wrist"])

In [ ]:
import gc, json, time
import numpy as np, pandas as pd, torch, timm
from pathlib import Path
from torch.utils.flop_counter import FlopCounterMode

SIZE = 384
assert torch.cuda.is_available(), "Active le GPU : Exécution > Modifier le type d'exécution > T4"

# ---------- 1. Coût d'inférence (paramètres, GFLOPs, latence) ----------
def cout_modele(model, size=SIZE, n_warm=20, n_run=100):
    model = model.to("cuda").eval()
    res = {"params_M": sum(p.numel() for p in model.parameters()) / 1e6}
    x = torch.randn(1, 3, size, size, device="cuda")
    with torch.no_grad(), FlopCounterMode(display=False) as fc:
        model(x)
    res["GFLOPs"] = fc.get_total_flops() / 1e9
    for bs in (1, 32):
        x = torch.randn(bs, 3, size, size, device="cuda")
        ts = []
        with torch.no_grad():
            for _ in range(n_warm):
                model(x)
            torch.cuda.synchronize()
            for _ in range(n_run):
                t0 = time.perf_counter()
                model(x)
                torch.cuda.synchronize()
                ts.append((time.perf_counter() - t0) * 1000)
        res[f"lat_ms_bs{bs}"] = float(np.median(ts))
    return res

specs = {
    "ResNet-50":    ("resnet50.tv_in1k", {}),
    "DenseNet-121": ("densenet121.tv_in1k", {}),
    "Swin-T":       ("swin_tiny_patch4_window7_224.ms_in22k_ft_in1k", dict(img_size=SIZE, window_size=12)),
    "ConvNeXt-T":   ("convnext_tiny.fb_in22k_ft_in1k", {}),
}

lignes = []
for nom, (arch, kw) in specs.items():
    reps = []
    for _ in range(3):                      # 3 répétitions, on garde la médiane
        gc.collect(); torch.cuda.empty_cache()
        m = timm.create_model(arch, pretrained=False, num_classes=1, **kw)
        reps.append(cout_modele(m))
        del m
    r = {k: float(np.median([x[k] for x in reps])) for k in reps[0]}
    r["lat_bs1_min"] = min(x["lat_ms_bs1"] for x in reps)
    r["lat_bs1_max"] = max(x["lat_ms_bs1"] for x in reps)
    r["modele"] = nom
    lignes.append(r)
    print(nom, "OK")
df_inf = pd.DataFrame(lignes).set_index("modele").round(2)

# ---------- 2. Temps d'entraînement + AUC, lus dans les result.json ----------
dossiers = {
    "ResNet-50":    [BASE / "runs_base" / f"expresnet50_seed{s}" for s in SEEDS],
    "DenseNet-121": [BASE / "runs_base" / f"expdensenet121_seed{s}" for s in SEEDS],
    "Swin-T":       [BASE / "runs_base" / f"expswin_t_seed{s}" for s in SEEDS],
    "ConvNeXt-T":   [BASE / "runs_v2" / f"expA_seed{s}" for s in SEEDS],
}
rows = []
for nom, ds in dossiers.items():
    for d in ds:
        f = d / "result.json"
        if not f.exists():
            print("MANQUANT :", f)
            continue
        r = json.load(open(f))
        rows.append({
            "modele": nom, "seed": int(d.name.split("seed")[-1]),
            "AUC_val": r["val"]["auc"], "AUC_test": r["test"]["auc"],
            "epochs": r["epochs_run"], "train_min": r["train_sec"] / 60,
            "s_par_epoch": r["train_sec"] / r["epochs_run"],
            "gpu": r.get("versions", {}).get("gpu", "?"),
        })
df_runs = pd.DataFrame(rows)
df_runs.to_csv(BASE / "resultats_par_graine.csv", index=False)

resume = df_runs.groupby("modele").agg(
    AUC_val_moy=("AUC_val", "mean"),
    AUC_test_moy=("AUC_test", "mean"), AUC_test_std=("AUC_test", "std"),
    epochs_moy=("epochs", "mean"), train_min_moy=("train_min", "mean"),
    s_par_epoch=("s_par_epoch", "mean"), n_graines=("seed", "count"),
).round(4)

# ---------- 3. Tableau final ----------
final = df_inf.join(resume)
final.to_csv(BASE / "tableau_final_modeles.csv")
print("GPU des entraînements :", df_runs.groupby("modele")["gpu"].unique().to_dict())
print("GPU de cette mesure   :", torch.cuda.get_device_name(0))
display(final)